In [ ]:
import sys, os, pygmt, importlib
mod_path = '/home/581/da1339/AFIM/src/AFIM/src'
sys.path.insert(0, mod_path)
from sea_ice_toolbox         import SeaIceToolbox, SeaIceToolboxManager
import numpy                 as np
import pandas                as pd
import xarray                as xr
import xesmf                 as xe
import matplotlib.pyplot     as plt
from pathlib                 import Path

In [ ]:
load_vars = ["aice", "tarea", "hi", "uvel", "vvel", "strength",
             "dvidtt", "daidtt", "dvidtd", "daidtd",
             "KuxE", "KuxN", "KuyE", "KuyN",
             "earea", "narea", "uarea"]
P_JSON = Path.home() / "AFIM" / "src" / "AFIM" / "src" / "JSONs" / "sea_ice_access-om3.json"
P_log = Path.home() / "logs" / f"LD_testing.log"
mgr   = SeaIceToolboxManager(P_log = P_log)
tb  = mgr.get_toolbox(P_json         = P_JSON,
                      sim_name       = "LD-om3-exp02", 
                      dt0_str        = "1993-01-01",
                      dtN_str        = "1993-12-31",
                      list_of_BorC2T = "Tc",
                      ice_type       = "FI",
                      hemisphere     = "south")
CICE_SO = tb.load_cice_zarr(slice_hem=True, variables=load_vars)
SIA_SH = tb.compute_hemisphere_ice_area(CICE_SO['aice'], CICE_SO['tarea'],
                                        ice_area_scale=tb.SIC_scale,
                                        add_grounded_iceberg_area=False)
tb.define_ice_mask_name(ice_type="FI")
I_day = tb.load_classified_ice(class_method="raw")[tb.mask_name]
I_bin = tb.load_classified_ice(class_method="binary-days")[tb.mask_name]

In [ ]:
A = CICE_SO["tarea"].isel(time=0)
# Apply mask(s)
I_daily = CICE_SO.where(I_day)
I_binly = CICE_SO.where(I_bin)
FIA = tb.compute_hemisphere_ice_area(I_binly['aice'], I_binly['tarea'],
                               ice_area_scale=tb.FIC_scale,
                               add_grounded_iceberg_area=False)
FIA.plot()
# # Build dict(s)
# I_dy = tb.metrics_data_dict(I_day, I_daily, A)
# I_bn = tb.metrics_data_dict(I_bin, I_binly, A)
# tb.define_metrics_zarr(class_method="raw")
# tb.compute_sea_ice_metrics(I_dy, tb.D_mets_zarr)
# tb.define_metrics_zarr(class_method="binary-days")
# tb.compute_sea_ice_metrics(I_bn, tb.D_mets_zarr)

In [ ]:
FIT = tb.compute_hemisphere_ice_thickness(I_binly['aice'], I_binly['hi'], I_binly['tarea'])
FIT.plot()

In [ ]:
SIA = tb.compute_hemisphere_ice_area(CICE_SO['aice'], CICE_SO['tarea'],
                               ice_area_scale=tb.SIC_scale,
                               add_grounded_iceberg_area=False)
SIA.plot()

In [ ]:
SIT = tb.compute_hemisphere_ice_thickness(CICE_SO['aice'], CICE_SO['hi'], CICE_SO['tarea'])
SIT.plot()